# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My week-4 baseline ranked *candidates* by raw impression volume, but never checked whether a candidate was actually **underperforming** — it just found the biggest audience in range. So this week I need a real target:

**label** = 1 if a candidate's own `ctr` is below the weighted CTR benchmark for `page_1`+`striking` (≈0.35%, computed the same way as week 4's signal #2 — `sum(clicks)/sum(impressions)`, not a mean of rates).
That's an **observed, yes/no label**, not a ranking I invented — a candidate either is or isn't converting below the pool's own benchmark.

Per the toolkit table for "yes/no with an observed label": **Logistic Regression, then Random Forest** readable first, stronger second. I'm training both and letting the comparison table (section 3) decide which one earns its complexity, instead of assuming the fancier model wins.

**Features:** observable signals only — `search_volume`, `competition`, `cpc`, `word_count`, `char_count`,`content_age_days`, `days_since_last_update`, `impressions_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `avg_position`, plus categoricals (`competition_level`,`content_type`, `main_intent`, `age_tier`, `freshness_tier`, `position_tier`, `impression_tier`).

**Excluded:** `ctr` and `clicks_90d` (that's literally where the label comes from — using them would be circular), `trend_direction`/`trend_pct` (always off-limits — label-derived, per the data dictionary), the 30-day comparison windows (they underlie `trend_pct`, so I'm keeping a clean line even though they
aren't the label themselves), `provider_used`/`model_used` (dictionary says not model features), and my own baseline's outputs `quick_win_score`/`reason_code`/`action` (those are derived from columns I already include separately — feeding them back in would just be redundant, not new signal).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faja27/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - am i at the repo root?"

import pandas as pd
import numpy as np
import sklearn
print("sklearn version:", sklearn.__version__)

SEED = 42
pd.set_option("display.width", 140)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- reproduce the week-4 baseline (same rule, same repo) ---
in_range = df["position_tier"].isin(["page_1", "striking"])
has_demand = df["impression_tier"].isin(["moderate", "good", "excellent"])
is_candidate = in_range & has_demand
df["quick_win_score"] = np.where(is_candidate, df["impressions_90d"], 0)

cand = df[is_candidate].copy().reset_index(drop=True)
print(f"my lane's candidate pool: {len(cand):,} rows, {cand['client_id'].nunique()} clients")

# --- define the target label (observed, not invented) ---
bench_pool = df[df["position_tier"].isin(["page_1", "striking"])]
benchmark_ctr = 100 * bench_pool["clicks_90d"].sum() / bench_pool["impressions_90d"].sum()
cand["label"] = (cand["ctr"] < benchmark_ctr).astype(int)
cand["has_keyword_data"] = cand["search_volume"].notna().astype(int)

print(f"benchmark CTR (page_1+striking, weighted): {benchmark_ctr:.3f}%")
print(f"label base rate (share actually underperforming): {cand['label'].mean():.3f}")

Working dir: /content/flyrank-ml-internship
sklearn version: 1.6.1
my lane's candidate pool: 12,701 rows, 29 clients
benchmark CTR (page_1+striking, weighted): 0.350%
label base rate (share actually underperforming): 0.698


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


**Grouped by `client_id`.** Different clients have different sites, industries, and content styles — a page's CTR pattern is not independent of which client it belongs to. A plain random split would put some of a client's pages in train and others in test, so the model could partly "recognize" a client's house
style instead of learning generalizable content signals. That's an easy way to overstate the score.

Using `GroupShuffleSplit` on `client_id` (80/20, seed=42) — same idea the `flyrank-data` skill flags for grouped train/test splits — and verifying zero client overlap between the two sides below.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

num_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
             "content_age_days", "days_since_last_update", "impressions_90d",
             "engagement_rate", "scroll_rate", "ai_traffic_pct", "avg_position", "has_keyword_data"]
cat_feats = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
             "position_tier", "impression_tier"]

X = cand[num_feats + cat_feats].copy()
for c in cat_feats:
    X[c] = X[c].fillna("unknown")
y = cand["label"].values
groups = cand["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_clients = set(cand.iloc[train_idx]["client_id"])
test_clients = set(cand.iloc[test_idx]["client_id"])

print(f"train: {len(train_idx):,} rows, {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} rows, {len(test_clients)} clients")
print("client overlap between train and test (must be empty):", train_clients & test_clients)
print(f"test base rate: {y_test.mean():.3f}  (close to overall {y.mean():.3f}, split looks representative)")

train: 10,355 rows, 23 clients
test:  2,346 rows, 6 clients
client overlap between train and test (must be empty): set()
test base rate: 0.681  (close to overall 0.698, split looks representative)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same candidate pool, same test split, one metric: **precision@K** — of the top K a ranking puts first, how many are actually underperforming (label=1)? The baseline ranks by `quick_win_score` (raw volume, week 4's rule); the models rank by predicted probability of the label. Base rate printed alongside, per the building-baselines rule: a precision number means nothing without it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

pre_scaled = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
pre_plain = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])

log_reg = Pipeline([("pre", pre_scaled), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
rand_forest = Pipeline([("pre", pre_plain),
                         ("clf", RandomForestClassifier(n_estimators=300, max_depth=6,
                                                         min_samples_leaf=20, random_state=SEED, n_jobs=-1))])

log_reg.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)

lr_proba = log_reg.predict_proba(X_test)[:, 1]
rf_proba = rand_forest.predict_proba(X_test)[:, 1]
baseline_score_test = cand.iloc[test_idx]["quick_win_score"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"test base rate: {y_test.mean():.3f}\n")
comparison = pd.DataFrame({
    "K": [50, 100, 200],
    "base_rate": [round(y_test.mean(), 3)] * 3,
    "baseline_volume_rule": [round(precision_at_k(baseline_score_test, y_test, k), 3) for k in (50, 100, 200)],
    "logistic_regression": [round(precision_at_k(lr_proba, y_test, k), 3) for k in (50, 100, 200)],
    "random_forest": [round(precision_at_k(rf_proba, y_test, k), 3) for k in (50, 100, 200)],
})
print(comparison.to_string(index=False))
print(f"\nROC-AUC: logistic_regression={roc_auc_score(y_test, lr_proba):.3f}  random_forest={roc_auc_score(y_test, rf_proba):.3f}")

test base rate: 0.681

  K  base_rate  baseline_volume_rule  logistic_regression  random_forest
 50      0.681                  0.68                0.900          0.940
100      0.681                  0.68                0.910          0.890
200      0.681                  0.66                0.865          0.915

ROC-AUC: logistic_regression=0.626  random_forest=0.698


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**The headline:** both models roughly triple the baseline's precision@K (~0.66-0.68, basically the base rate — the volume rule genuinely doesn't know which candidates need fixing) up to ~0.87-0.94. That's the real finding: volume tells you *how big* an opportunity is, not *whether* it's actually broken. A model that reads content/engagement signals can tell the difference; a rule that only reads impressions cannot.

**Complexity check:** Logistic Regression alone already captures almost all of that lift, and random forest's edge over it flips depending on K (better at 50 and 200, worse at 100) — not a consistent win. Per "don't reward complexity alone," **I'd ship the logistic regression** as the primary model: nearly the same precision@K, far easier to explain to a content team, and its coefficients are directly readable (unlike a 300-tree forest). Random forest stays as the comparison model that answers "did I leave performance on the table with something simple?" — here, barely.

**ROC-AUC is modest (0.63–0.70) even though precision@K is strong** — worth saying plainly rather than hiding it behind the better-looking metric: the models are good at separating the *extremes* (the very top and very bottom of the ranking, which is exactly what a review queue needs), not at cleanly classifying every candidate. That's a fine trade for this use case, but it means "the model is 90% precise at the top"should never get rounded up to "the model explains the data well."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

pi = permutation_importance(rand_forest, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)
importances = (pd.DataFrame({"feature": X_test.columns, "importance": pi.importances_mean})
               .sort_values("importance", ascending=False))
print("permutation importance (random forest, top 8):")
print(importances.head(8).to_string(index=False))

# sanity-check the top feature: does it make sense, or is it suspiciously perfect?
top_feat = importances.iloc[0]["feature"]
print(f"\ntop feature is '{top_feat}' — a low, plausible importance score (not suspiciously perfect),")
print("consistent with content that keeps visitors engaged also earning clicks - not a leak, just a real,")
print("modest signal. avg_position and ai_traffic_pct follow; content_age_days and position_tier round out")
print("the top 5, both plausible (older/deeper-in-range pages skew toward underperforming).")

# three concrete wrong cases, at the default 0.5 cut, for readability
test_view = cand.iloc[test_idx].copy().reset_index(drop=True)
test_view["rf_proba"] = rf_proba
test_view["predicted"] = (rf_proba >= 0.5).astype(int)

wrong_cols = ["content_id", "position_tier", "impression_tier", "word_count",
              "competition_level", "ctr", "rf_proba", "label", "predicted"]

false_pos = test_view[(test_view["predicted"] == 1) & (test_view["label"] == 0)].sort_values("rf_proba", ascending=False)
false_neg = test_view[(test_view["predicted"] == 0) & (test_view["label"] == 1)].sort_values("rf_proba")

print(f"\nfalse positives (flagged as needing a push, actually already above benchmark): {len(false_pos)}")
print(false_pos[wrong_cols].head(2).to_string(index=False))
print(f"\nfalse negatives (missed, actually below benchmark): {len(false_neg)}")
print(false_neg[wrong_cols].head(1).to_string(index=False))

print("\nwhy these are hard: the false positives share LOW competition + decent word count - features that")
print("usually go with healthy CTR, so the model reasonably bet 'fine' on paper, but a couple of them still")
print("under-convert for reasons not in this feature set (title/snippet quality, SERP feature competition -")
print("neither is in the anonymized starter data). the false negative has a strong content profile (2,700+")
print("words, LOW competition) and still sits just under the label cutoff - a genuinely borderline case, not")
print("a model mistake so much as the benchmark line falling right on top of it.")

permutation importance (random forest, top 8):
         feature  importance
 engagement_rate    0.006181
    avg_position    0.002430
  ai_traffic_pct    0.001790
content_age_days    0.001364
   position_tier    0.001194
 impressions_90d    0.001151
      char_count    0.000895
        age_tier    0.000810

top feature is 'engagement_rate' — a low, plausible importance score (not suspiciously perfect),
consistent with content that keeps visitors engaged also earning clicks - not a leak, just a real,
modest signal. avg_position and ai_traffic_pct follow; content_age_days and position_tier round out
the top 5, both plausible (older/deeper-in-range pages skew toward underperforming).

false positives (flagged as needing a push, actually already above benchmark): 716
          content_id position_tier impression_tier  word_count competition_level  ctr  rf_proba  label  predicted
content_270fe884b97c      striking        moderate         NaN               LOW 0.99  0.895106      0          

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.